[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RickBarretto/llm-playground/blob/main/play.ipynb)

## Setup

This notebook can test any branch or tag of `ricklm`.

Before running the model, enable a GPU runtime in Colab: `Runtime` -> `Change runtime type` -> `T4 GPU` or better.

Set `GIT_REF` to the branch or tag you want to test. Examples: `main`, `api-v2`, `v1.0.0`.

In [ ]:
GIT_REF = "api-v2"  # branch or tag

!rm -rf /content/llm-playground
!git clone https://github.com/RickBarretto/llm-playground /content/llm-playground
%cd /content/llm-playground
!git fetch --all --tags
!git checkout $GIT_REF
!pip install -U /content/llm-playground

## Set Model

|Model|Available Sizes|Default Size|Hugging Face|
|:---:|:--------------|:----------:|------------|
|`AmadeusVerbo`  |`0.5B`, `1.5B`, `3B`, `7B`, `14B`, `32B`, `72B`|`7B`|https://huggingface.co/collections/amadeusai/amadeus-verbo-qwen25-pt-br-powered-by-aws|
|`Gaia`          |`4B`|`4B`|https://huggingface.co/CEIA-UFG/Gemma-3-Gaia-PT-BR-4b-it|
|`Tucano`        |`1.1B`, `2.4B`| `2.4B`|https://huggingface.co/collections/TucanoBR/tucano|
|`TeenyTinyLlama`|`460m`| `460m`|https://huggingface.co/collections/nicholasKluge/teenytinyllama|

**Usage:**

```python
from ricklm import models

model = models.AmadeusVerbo("3B")
```

In [ ]:
from ricklm import models

model = models.AmadeusVerbo("3B")

## Ask

`ask()` returns the model answer as normalized UTF-8 text when printed or written to disk.

Minimal Colab test:

```python
from pathlib import Path
from ricklm import models

model = models.AmadeusVerbo("3B")

with model as m:
    output = m.ask("Escreva uma frase curta sobre o Brasil.")
    print(output)
    Path("/content/ask.txt").write_text(output, encoding="utf-8")
```

In [ ]:
from pathlib import Path

OUTPUT_DIR = Path("/content/llm-playground/outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with model as m:
    output = m.ask("Escreva uma frase curta sobre o Brasil.")
    print(output)
    (OUTPUT_DIR / "ask.txt").write_text(output, encoding="utf-8")


## Chat

`chat()` yields turns in a natural format:

```text
Usuário:

...

<Modelo>:

...
```

Use `sair`, `exit`, or `quit` to finish an interactive chat.

In [ ]:
with model as m:
    for output in m.chat():
        print(output)
        print()

    (OUTPUT_DIR / "chat-last.txt").write_text(output, encoding="utf-8")
    


## Chat History

A chat history can be indexed. `history[-1]` prints only the last user/model turn.

Use `history[-1].ai` or `history.last.ai` to get only the model answer.

This is the minimal scripted chat test for Colab.

In [ ]:
with model as m:
    history = m.chat([
        "Escreva uma frase sobre o Brasil.",
        "Agora transforme essa frase em um verso."
    ])

    print(history[-1])
    print(history[-1].ai)
    print(history.last.ai)
    (OUTPUT_DIR / "chat-last.txt").write_text(history[-1], encoding="utf-8")


### Push to GitHub

Commits generated outputs and pushes to the repo.

Set `GITHUB_TOKEN` and pass your git username and email to push into your repository.

To set secrets: (key sidebar -> Secrets).

**Usage**

```py
from ricklm.features import GitHub, Path
from ricklm.features import secret

GitHub(
    repo="Username/Repo",
    token=secret("GITHUB_TOKEN"),
    username=secret("GIT_NAME"),
    email=secret("GIT_EMAIL"),
    branch="your-branch",
    root=REPO_ROOT
).push("Your commit message")
```

In [ ]:
from ricklm.features import GitHub, Path
from ricklm.features import secret

REPO_ROOT = Path("/content/llm-playground")

GitHub(
    repo="RickBarretto/llm-playground",
    token=secret("GITHUB_TOKEN"),
    username=secret("GIT_NAME"),
    email=secret("GIT_EMAIL"),
    branch="main",
    root=REPO_ROOT
).push(f"[Outputs] Publish generated text by {model}.")

If you want to reuse this same runtime, make sure to `model.release()` and `del model`.

This ensures the storage will be clean and also the model will be garbage collected.

In [ ]:
model.release()
del model